# dMRI preprocessing QC - single-subject demo

This notebook shows quality-control (QC) metrics and visualizations for **one subject**
from the preprocessing stage (`den_gr`, `topup`, `eddy`, `bias`).

It does not run FSL/MRtrix/ANTs: it **only reads** outputs that those steps already
wrote to disk and turns them into tables (`pandas`) and figures (`matplotlib`) using
the `preprocessing_qc` subpackage.

**Flow:** acquisition shells -> denoising QC (SNR + before/after) -> eddy QC
(motion, outliers) -> `eddy_quad` summary (CNR) -> subject summary table ->
cohort summary (multiple subjects) -> CSV.


## 1. Configuration

Edit `BASE` and `SUB`. Paths follow the naming convention used by the preprocessing scripts.

In [ ]:
import sys
from pathlib import Path

# Allow imports whether the notebook is launched from notebooks/ or the repo root.
for cand in [Path.cwd() / "src", Path.cwd(), Path.cwd().parent / "src", Path.cwd().parent]:
    if (cand / "preprocessing_qc").is_dir():
        sys.path.insert(0, str(cand))
        break

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import preprocessing_qc as qc

pd.set_option("display.float_format", lambda v: f"{v:,.3f}")


In [ ]:
# --- EDIT THIS ---
BASE = Path("/mnt/storage/tier2/MUMI-EXT-001/mumi-data/USERS/GonzAny/"
            "MRI-processing/MRI-Lab-Lisa-dMRI/data")
SUB = "c01"

dwi_dir  = BASE / SUB / "ses-T0" / "dwi"
den_dir  = dwi_dir / "den"
eddy_out = den_dir / "preproc-2" / "eddy-out"
topup_out = den_dir / "preproc-1" / "topup-out"

# Files, using the names generated by the current scripts.
raw_dwi     = dwi_dir / f"{SUB}_ses-T0_dwi.nii.gz"
bval        = dwi_dir / f"{SUB}_ses-T0_dwi.bval"
den_dwi     = den_dir / f"{SUB}_ses-T0_dwi_den.nii.gz"
den_grc     = den_dir / f"{SUB}_ses-T0_dwi_den_grc.nii.gz"
noise_map   = den_dir / "denoising-der" / f"{SUB}_ses-T0_dwi_noise.nii.gz"
residuals   = den_dir / "denoising-der" / f"{SUB}_ses-T0_dwi_den-residuals.nii.gz"
mask        = topup_out / f"{SUB}_ses-T0_dwi_topup-out-preproc-1-unwarp-imgs-mean_brain-f0.3R_mask.nii.gz"

# Eddy basename (without suffix): .eddy_movement_rms, .qc/qc.json, etc. hang from here.
eddy_base   = eddy_out / f"{SUB}_ses-T0_dwi_den_grc_tec_preproc-2"
corrected   = Path(str(eddy_base) + ".nii.gz")

print("eddy_base:", eddy_base)


## 2. Acquisition Scheme (Shells)

How many directions exist for each b-value. Useful for checking that the protocol is the expected one.

In [ ]:
shells = qc.summarize_shells(bval)
display(shells)
qc.plot_shells(shells)
plt.show()


## 3. Denoising QC

b=0 SNR using the `dwidenoise` noise map (`sigma`), plus a visual before/after comparison. The *Difference* image should show noise without anatomical structure.

In [ ]:
snr = qc.compute_denoising_snr(raw_dwi, noise_map, bval, mask_path=mask)
display(snr.to_frame("value"))

qc.plot_before_after_slice(raw_dwi, den_dwi, titles=("Raw", "Denoised"), volume=0)
plt.show()


In [ ]:
# Residuals: mean ~ 0 and no structure => good denoising.
res = qc.compute_residual_stats(residuals, mask_path=mask)
display(res.to_frame("value"))


## 4. Eddy QC: Motion and Outliers

Motion RMS by volume, translation/rotation parameters, and the map of slices marked as outliers by `--repol`.

In [ ]:
mrms = qc.load_movement_rms(str(eddy_base) + ".eddy_movement_rms")
display(mrms.describe().loc[["mean", "max"]])
qc.plot_movement_rms(mrms)
plt.show()


In [ ]:
motion = qc.load_motion_parameters(str(eddy_base) + ".eddy_parameters")
qc.plot_motion_parameters(motion)
plt.show()


In [ ]:
omap = qc.load_outlier_map(str(eddy_base) + ".eddy_outlier_map")
display(qc.summarize_outliers(omap).to_frame("value"))
qc.plot_outlier_heatmap(omap)
plt.show()


### Eddy Before/After

Comparison between the eddy input (`den_grc`) and corrected output on one diffusion-weighted volume.

In [ ]:
qc.plot_before_after_slice(den_grc, corrected, titles=("den_grc", "eddy-corr"), volume=30)
plt.show()


## 5. `eddy_quad` Summary (qc.json)

Metrics summarized by `eddy_quad`: mean motion, outlier percentage, and CNR by shell.

In [ ]:
qcjson = qc.load_qc_json(str(eddy_base) + ".qc/qc.json")
display(qc.qc_json_to_series(qcjson).to_frame("value"))
qc.plot_cnr_per_shell(qcjson)
plt.show()


## 6. Subject Summary Table

Everything above condensed into **one `pandas` row**, the unit that is later stacked across the cohort.

In [ ]:
summary = qc.subject_qc_summary(
    SUB, eddy_base,
    dwi_path=raw_dwi, noise_path=noise_map, residuals_path=residuals,
    bval_path=bval, mask_path=mask,
)
summary.T


## 7. Cohort Summary (Multiple Subjects) -> CSV

The pipeline processes **all** images in a data folder. Here we build a table
with one subject per row by scanning that folder, then compare one metric across
subjects to detect atypical cases. Edit the `SUBJECTS` list.

In [ ]:
SUBJECTS = ["c01"]  # e.g. ["c01", "c02", "c03", ...]

specs = []
for s in SUBJECTS:
    eb = BASE / s / "ses-T0" / "dwi" / "den" / "preproc-2" / "eddy-out" / \
         f"{s}_ses-T0_dwi_den_grc_tec_preproc-2"
    dv = BASE / s / "ses-T0" / "dwi"
    dn = dv / "den"
    specs.append(dict(
        subject_id=s, eddy_base=eb,
        dwi_path=dv / f"{s}_ses-T0_dwi.nii.gz",
        noise_path=dn / "denoising-der" / f"{s}_ses-T0_dwi_noise.nii.gz",
        bval_path=dv / f"{s}_ses-T0_dwi.bval",
    ))

cohort = qc.cohort_qc_summary(specs)
display(cohort)

# Between-subject comparison. Choose the column you want to audit.
if len(cohort) and "mot_abs_mm_mean" in cohort.columns:
    qc.plot_cohort_metric(cohort, "mot_abs_mm_mean", ylabel="mean absolute motion (mm)")
    plt.show()

cohort.to_csv("preprocessing_qc_summary.csv")
print("saved -> preprocessing_qc_summary.csv")


---
### AI Use Note

Parts of this notebook and the `preprocessing_qc` package were developed with AI
assistance for code and documentation refinement. Metric design, result
interpretation, and validation on data remain the author's responsibility.
